In [ ]:
from pypdf import PdfReader
import warnings
warnings.filterwarnings("ignore")
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np
import time
import math
import chromadb
from chromadb.config import Settings

from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import re

In [2]:
pdf_path = "Pensamento_maquina.pdf"

reader = PdfReader(pdf_path)

print(f"Numéro de páginas : {len(reader.pages)}")


Numéro de páginas : 4


In [3]:
full_text = ""

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    full_text += text + "\n"

print("Tamanho total do texto:", len(full_text))

Tamanho total do texto: 9049


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     
    chunk_overlap=150,  
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_text(full_text)

print("Número total de chunks:", len(chunks))

for i, chunk in enumerate(chunks[:25]):
    print("\n====================")
    print(f"CHUNK {i}")
    print("====================")
    print(chunk)


Número total de chunks: 25

CHUNK 0
O Pensamento Humano e o Pensamento das 
Máquinas
Introdução
Desde os primórdios da ﬁlosoﬁa, o ser humano busca compreender a própria mente. 
Perguntas como o que é pensar?, como surge o conhecimento? e o que nos torna conscientes? 
atravessam séculos de reﬂexão ﬁlosóﬁca, cientíﬁca e cultural. Com o avanço da 
tecnologia e, especialmente, com o surgimento da Inteligência Artiﬁcial, essas questões 
ganharam uma nova dimensão: ao criar máquinas capazes de executar tarefas cognitivas,

CHUNK 1
ganharam uma nova dimensão: ao criar máquinas capazes de executar tarefas cognitivas, 
o ser humano passou a se perguntar se estaria, de alguma forma, reproduzindo o próprio 
pensamento.
A relação entre o pensamento humano e o pensamento das máquinas não é apenas 
técnica. Trata-se de uma questão conceitual, ﬁlosóﬁca e ética. Comparar esses dois tipos 
de “pensamento” nos obriga a reﬂetir sobre os limites da tecnologia e, ao mesmo tempo, 
sobre a natureza da mente 

In [5]:
st = SentenceTransformer("all-MiniLM-L6-v2")

print("Gerando embeddings dos chunks...\n")

chunk_embeddings = st.encode(chunks)

print("Número de embeddings:", len(chunk_embeddings))
print("Shape de UM embedding:", chunk_embeddings[0].shape)

print("\nPrimeiros 10 números do embedding do CHUNK 0:")
print(chunk_embeddings[0][:10])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1091.88it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Gerando embeddings dos chunks...

Número de embeddings: 25
Shape de UM embedding: (384,)

Primeiros 10 números do embedding do CHUNK 0:
[ 0.02200047  0.11847519 -0.00923042  0.01974403 -0.07060358  0.00535173
  0.03218771  0.07159293  0.02006063  0.07255962]


Implementação de sistema de Dense Retrieval baseado em embeddings semânticos e similaridade do cosseno para busca contextual em documentos.

In [6]:
queries = [
    "Explique como a IA usa matemática para processar informação",
    "Como redes neurais artificiais se inspiram no cérebro humano?",
    "Quais tarefas as máquinas realizam melhor que os humanos?"
]

top_k = 3

for idx, query in enumerate(queries, start=1):

    query_embedding = st.encode([query])
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    print(f"\n==============================")
    print(f"Query {idx}: {query}")

    for rank, chunk_idx in enumerate(top_indices, start=1):
        print(f"\nTop {rank}")
        print(f"Chunk index: {chunk_idx}")
        print(f"Similaridade: {similarities[chunk_idx]:.4f}")
        print(f"{chunks[chunk_idx][:500]}...\n")



Query 1: Explique como a IA usa matemática para processar informação

Top 1
Chunk index: 8
Similaridade: 0.5838
Artiﬁcial são projetados para processar grandes volumes de dados, identiﬁcar padrões e 
produzir respostas ou ações com base nesses padrões.
Diferentemente do ser humano, a máquina não possui consciência, emoções ou intenções 
próprias. Quando uma IA “decide” algo, essa decisão é o resultado de cálculos 
matemáticos realizados a partir de parâmetros deﬁnidos durante o treinamento. Não há 
compreensão subjetiva do que está sendo feito, apenas a execução de regras estatísticas....


Top 2
Chunk index: 24
Similaridade: 0.5686
como uma colaboração — uma parceria entre o pensamento consciente e o cálculo 
automatizado....


Top 3
Chunk index: 3
Similaridade: 0.5469
A natureza do pensamento humano
O pensamento humano é um fenômeno extremamente complexo. Ele não pode ser 
reduzido a uma sequência linear de operações lógicas, pois envolve múltiplas dimensões 
que interagem entre si 

In [7]:
client = chromadb.Client()

collection = client.create_collection(name="rag_test")

print("Collection criada.")


Collection criada.


In [8]:
ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    ids=ids
)

print("Chunks inseridos no ChromaDB.")
print("Total armazenado:", len(ids))


Chunks inseridos no ChromaDB.
Total armazenado: 25


In [9]:


device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_llm = AutoTokenizer.from_pretrained(model_name)

model_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [01:51<00:00,  2.61it/s, Materializing param=model.norm.weight]                               


In [10]:
def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)

def f1(p, r):
    if p + r == 0:
        return 0
    return 2 * (p * r) / (p + r)

def mrr(retrieved, relevant):
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            return 1 / rank
    return 0

def ndcg_at_k(retrieved, relevant, k):
    dcg = 0
    for i in range(len(retrieved[:k])):
        if retrieved[i] in relevant:
            dcg += 1 / math.log2(i + 2)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / math.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0


queries = [
    "Explique como a IA usa matemática para processar informação",
    "Como redes neurais artificiais se inspiram no cérebro humano?",
    "Quais tarefas as máquinas realizam melhor que os humanos?"
]

k = 3

ground_truth = {
    0: ["chunk_8", "chunk_9", "chunk_10"],
    1: ["chunk_11", "chunk_12"],
    2: ["chunk_19", "chunk_15"]
}


all_precisions, all_recalls, all_f1s, all_mrrs, all_ndcgs, latencies = [], [], [], [], [], []

for q_idx, query in enumerate(queries):

    start_time = time.time()

    query_embedding = st.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k
    )

    retrieved_ids = results["ids"][0]

    latency = time.time() - start_time
    latencies.append(latency)

    relevant_ids = ground_truth[q_idx]

    p = precision_at_k(retrieved_ids, relevant_ids, k)
    r = recall_at_k(retrieved_ids, relevant_ids, k)
    f = f1(p, r)
    m = mrr(retrieved_ids, relevant_ids)
    n = ndcg_at_k(retrieved_ids, relevant_ids, k)

    all_precisions.append(p)
    all_recalls.append(r)
    all_f1s.append(f)
    all_mrrs.append(m)
    all_ndcgs.append(n)

    print("\n==============================")
    print("Query:", query)
    print("Retrieved IDs:", retrieved_ids)
    print("Chunks:")
    for doc in results["documents"][0]:
        print("-", doc[:200], "...")

    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}: {r:.4f}")
    print(f"F1@{k}: {f:.4f}")
    print(f"MRR: {m:.4f}")
    print(f"nDCG@{k}: {n:.4f}")
    print(f"Tempo de resposta: {latency:.6f} segundos")


print("\n==============================")
print("RESULTADOS FINAIS (CHUNKS)")
print("==============================")
print(f"Mean Precision@{k}: {np.mean(all_precisions):.4f}")
print(f"Mean Recall@{k}: {np.mean(all_recalls):.4f}")
print(f"Mean F1@{k}: {np.mean(all_f1s):.4f}")
print(f"Mean MRR: {np.mean(all_mrrs):.4f}")
print(f"Mean nDCG@{k}: {np.mean(all_ndcgs):.4f}")
print(f"Latência média: {np.mean(latencies):.6f} segundos")



Query: Explique como a IA usa matemática para processar informação
Retrieved IDs: ['chunk_8', 'chunk_24', 'chunk_3']
Chunks:
- Artiﬁcial são projetados para processar grandes volumes de dados, identiﬁcar padrões e 
produzir respostas ou ações com base nesses padrões.
Diferentemente do ser humano, a máquina não possui consciên ...
- como uma colaboração — uma parceria entre o pensamento consciente e o cálculo 
automatizado. ...
- A natureza do pensamento humano
O pensamento humano é um fenômeno extremamente complexo. Ele não pode ser 
reduzido a uma sequência linear de operações lógicas, pois envolve múltiplas dimensões 
que i ...
Precision@3: 0.3333
Recall@3: 0.3333
F1@3: 0.3333
MRR: 1.0000
nDCG@3: 0.4693
Tempo de resposta: 10.499678 segundos

Query: Como redes neurais artificiais se inspiram no cérebro humano?
Retrieved IDs: ['chunk_11', 'chunk_12', 'chunk_2']
Chunks:
- não entende signiﬁcados; ela manipula representações numéricas associadas a padrões 
observados.
A inspiração no cé

In [16]:
def generate_answer(prompt):

    messages = [
        {"role": "system", "content": "Responda sempre em português e seja objetivo."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer_llm.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer_llm(
        text,
        return_tensors="pt"
    )

    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer_llm.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()


In [12]:
def mini_rag(query, n_results=3):

    query_embedding = st.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]

    context = "\n".join(retrieved_chunks[:2])

    prompt = f"""
Use apenas o contexto para responder.

Contexto:
{context}

Pergunta:
{query}
"""

    answer = generate_answer(prompt)

    return {
        "query": query,
        "context": context,
        "answer": answer
    }


In [ ]:
generate_answer("Me diga 3 principais diferenças entre o cérebro humano e o processamento das máquinas, seja breve")

In [13]:
def extrair_json(texto):
    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None


def avaliar_relevancia(pergunta, resposta):

    prompt = f"""
Avalie de 0 a 10 o quanto a resposta responde corretamente a pergunta.

Pergunta:
{pergunta}

Resposta:
{resposta}

Retorne apenas JSON:
{{"relevancia": numero}}
"""

    result = generate_answer(prompt)
    return extrair_json(result)



def avaliar_fidelidade(contexto, resposta):

    prompt = f"""
Avalie de 0 a 10 se a resposta usa apenas informações do contexto.
Se inventar coisas, reduza a nota.

Contexto:
{contexto}

Resposta:
{resposta}

Retorne apenas JSON:
{{"fidelidade": numero}}
"""

    result = generate_answer(prompt)
    return extrair_json(result)


def avaliar_correcao(pergunta, resposta):

    prompt = f"""
Avalie de 0 a 10 se a resposta está factualmente correta
com base em conhecimento geral.

Pergunta:
{pergunta}

Resposta:
{resposta}

Retorne apenas JSON:
{{"correcao": numero}}
"""

    result = generate_answer(prompt)
    return extrair_json(result)



def avaliar_completude(pergunta, resposta):

    prompt = f"""
Avalie de 0 a 10 se a resposta está completa
ou se faltam partes importantes.

Pergunta:
{pergunta}

Resposta:
{resposta}

Retorne apenas JSON:
{{"completude": numero}}
"""

    result = generate_answer(prompt)
    return extrair_json(result)


def avaliar_resposta_rag(pergunta, contexto, resposta):

    return {
        "relevancia": avaliar_relevancia(pergunta, resposta),
        "fidelidade": avaliar_fidelidade(contexto, resposta),
        "correcao": avaliar_correcao(pergunta, resposta),
        "completude": avaliar_completude(pergunta, resposta)
    }


In [14]:
resultado = mini_rag("Explique redes neurais")

avaliacao = avaliar_resposta_rag(
    pergunta=resultado["query"],
    contexto=resultado["context"],
    resposta=resultado["answer"]
)

print(avaliacao)


{'relevancia': {'relevancia': 8}, 'fidelidade': {'fidelidade': 8}, 'correcao': {'correcao': 9}, 'completude': {'completude': 8.5}}
